<div style="font-size: 2em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Régression linéaire - RMSE public : 3.2852
</div>
<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des bilbiothèques
</div>

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
import gc
import numpy as np

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Importation des documents de données temporelles et de test
</div>

In [2]:
stations_toulouse = pd.read_parquet("stations_toulouse.parquet") # Données géographiques (Colonnes : 'station', 'longitude', 'latitude', 'station_id')
history_toulouse_train = pd.read_parquet("history_toulouse_train.parquet")
history_toulouse_test = pd.read_parquet("history_toulouse_test.parquet")

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Création d'un dataset de test pour évaluer sa performance
</div>
On récupère les 8 dernières valeurs

In [3]:
rang_depuis_la_fin = history_toulouse_train.groupby(level="station").cumcount(ascending=False)
history_toulouse_apprentissage = history_toulouse_train[rang_depuis_la_fin >= 8].copy()

On créé l'ensemble de validation avec la dernière valeur connue + les 8 à prédire

In [4]:
last_9 = history_toulouse_train.groupby(level="station").tail(9).copy()

On remet l'index sous forme de colonnes

In [5]:
last_9 = last_9.reset_index()

On calule la somme cummulée

In [6]:
last_9["horizon"] = last_9.groupby("station").cumcount()

On crée une ligne par station et on récupère les 8 valeurs futures

In [7]:
history_toulouse_validation = (
    last_9
    .pivot(
        index="station",
        columns="horizon",
        values="bikes"
    )
    .drop(columns=0)
    .rename(columns={
        1: "bikes_+15min",
        2: "bikes_+30min",
        3: "bikes_+45min",
        4: "bikes_+60min",
        5: "bikes_+75min",
        6: "bikes_+90min",
        7: "bikes_+105min",
        8: "bikes_+120min"
    })
)

Récupérer la date de la dernière mesure de l'ensemble d'apprentissage par station

In [8]:
cutoff_times = last_9[last_9["horizon"] == 0].set_index("station")["commit_at"]

Ajouter commit_at

In [9]:
history_toulouse_validation["commit_at"] = cutoff_times

Mettre station + commit_at en MultiIndex

In [10]:
history_toulouse_validation = history_toulouse_validation.reset_index().set_index(["station", "commit_at"]).sort_index()

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Données du train
</div>
On transforme les indexe en colonnes pour plus de praticité

In [11]:
train = history_toulouse_train.reset_index()

Prise en compte de la date

In [12]:
train["annee"] = train["commit_at"].dt.year
train["mois"] = train["commit_at"].dt.month
train["jour"] = train["commit_at"].dt.dayofweek
train["heure"] = train["commit_at"].dt.hour
train["minute"] = train["commit_at"].dt.minute

Prise en compte de la géolocalisation

In [13]:
train = train.merge(stations_toulouse, on="station", how="left")

Données utiles pour l'entrainement du modèle

In [14]:
colonnes_train = ["bikes", "annee", "mois", "jour", "heure", "minute", "longitude", "latitude"]

On extrait tous les indexe station et commit_at_index sous forme de liste

In [15]:
station_index = history_toulouse_test.index.get_level_values("station")
commit_at_index = history_toulouse_test.index.get_level_values("commit_at")

On crée un dataframe reliant (station, date) au nombre de vélos

In [16]:
date_velo = train.set_index(["station", "commit_at"])["bikes"]

On associe chaque nom de colonne à une valeur numérique

In [17]:
decalage = {
    "bikes_+15min": 15,
    "bikes_+30min": 30,
    "bikes_+45min": 45,
    "bikes_+60min": 60,
    "bikes_+75min": 75,
    "bikes_+90min": 90,
    "bikes_+105min": 105,
    "bikes_+120min": 120,
}

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Données du test
</div>
On récupère le nombre de vélos à l'instant zero du test

In [18]:
instant_zero = list(zip(station_index, commit_at_index))
bikes_train = history_toulouse_train["bikes"]

On localise chaque index (station, date) pour trouver le nombre de vélos à l'instant zero du test

In [19]:
bikes_zero = pd.Series(bikes_train.loc[instant_zero].values, index=history_toulouse_test.index)

On récupère la géolocalisation de chaque station du test <br>
On définit la colonne "station" comme étant l'indexe

In [20]:
stations_pos = stations_toulouse.set_index("station")

On localise chaque station pour trouver leur géolocalisation

In [21]:
stations_pos_test = stations_pos.reindex(station_index)
longitude_test = [stations_pos.loc[station, "longitude"] for station in station_index]
latitude_test = [stations_pos.loc[station, "latitude"] for station in station_index]

On crée un dataframe avec toutes les données du test dont on aura besoin pour prédire. <br>
Il faut qu'il y ait les mêmes données que celles utilisées pour l'entrainement

In [22]:
donnees_test = pd.DataFrame({
    "bikes": bikes_zero,
    "annee": commit_at_index.year,
    "mois": commit_at_index.month,
    "jour": commit_at_index.dayofweek,
    "heure": commit_at_index.hour,
    "minute": commit_at_index.minute,
    "longitude": longitude_test,
    "latitude": latitude_test,
}, index=history_toulouse_test.index)

<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Entrainement et prédictions
</div>
On entraine un modèle par décalage horaire et on le fait prédire

In [23]:
for col, h in decalage.items():
    print(h/1.2, "%")
    # On crée un vecteur avec l'horaire à prédire
    commit_at_cible = train["commit_at"] + pd.Timedelta(minutes=h)
    # On créé un dataset avec le nombre de vélos en prenant en compte le décalage horaire
    futur = train[["station", "commit_at", "bikes"]].rename(
        columns={"commit_at": "commit_at_cible", "bikes": "bikes_futur"}
    )
    # On 
    avec_futur = train.assign(commit_at_cible=commit_at_cible).merge(
        futur, on=["station", "commit_at_cible"], how="inner"
    )

    # Prédictions
    model = LinearRegression()
    # On renseigne les données d'entrainement et leur réponse
    X = avec_futur[colonnes_train]
    y = avec_futur["bikes_futur"]
    # On entraine le modèle
    model.fit(X, y)
    # On prédit et écrit les prédictions dans history_toulouse_test
    history_toulouse_test[col] = model.predict(donnees_test[colonnes_train]).round().astype(int)

    # On libère la mémoire avant l'itération suivante
    del commit_at_cible, futur, avec_futur, X, y, model
    gc.collect()

12.5 %
25.0 %
37.5 %
50.0 %
62.5 %
75.0 %
87.5 %
100.0 %


<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
On recalcule tout avec TOUTES les données d'entraînement fournies
</div>

In [24]:
train = history_toulouse_train.reset_index()
train["annee"] = train["commit_at"].dt.year
train["mois"] = train["commit_at"].dt.month
train["jour"] = train["commit_at"].dt.dayofweek
train["heure"] = train["commit_at"].dt.hour
train["minute"] = train["commit_at"].dt.minute
train = train.merge(stations_toulouse, on="station", how="left")
station_index = history_toulouse_test.index.get_level_values("station")
commit_at_index = history_toulouse_test.index.get_level_values("commit_at")
date_velo = train.set_index(["station", "commit_at"])["bikes"]
instant_zero = list(zip(station_index, commit_at_index))
bikes_train = history_toulouse_train["bikes"]
bikes_zero = pd.Series(bikes_train.loc[instant_zero].values, index=history_toulouse_test.index)
stations_pos = stations_toulouse.set_index("station")
stations_pos_test = stations_pos.reindex(station_index)
longitude_test = [stations_pos.loc[station, "longitude"] for station in station_index]
latitude_test = [stations_pos.loc[station, "latitude"] for station in station_index]
donnees_test = pd.DataFrame({
    "bikes": bikes_zero,
    "annee": commit_at_index.year,
    "mois": commit_at_index.month,
    "jour": commit_at_index.dayofweek,
    "heure": commit_at_index.hour,
    "minute": commit_at_index.minute,
    "longitude": longitude_test,
    "latitude": latitude_test,
}, index=history_toulouse_test.index)
for col, h in decalage.items():
    print(h/1.2, "%")
    commit_at_cible = train["commit_at"] + pd.Timedelta(minutes=h)
    futur = train[["station", "commit_at", "bikes"]].rename(columns={"commit_at": "commit_at_cible", "bikes": "bikes_futur"})
    avec_futur = train.assign(commit_at_cible=commit_at_cible).merge(futur, on=["station", "commit_at_cible"], how="inner")
    model = LinearRegression()
    X = avec_futur[colonnes_train]
    y = avec_futur["bikes_futur"]
    model.fit(X, y)
    history_toulouse_test[col] = model.predict(donnees_test[colonnes_train]).round().astype(int)
    del commit_at_cible, futur, avec_futur, X, y, model
    gc.collect()

12.5 %
25.0 %



KeyboardInterrupt



<div style="font-size: 1em; font-weight: bold; margin-top: 0.67em; margin-bottom: 0.67em;">
Sauvegarde du résultat
</div>

In [ ]:
history_toulouse_test.to_parquet("BA_prediction_100_RL.parquet", index=True)
history_toulouse_test